In [1]:
import pandas as pd
from pathlib import Path
import gc

from schema import SCHEMA_CURICA

In [2]:
# toma como diretório pai o diretório do arquivo do notebook
BASE_DIR = Path("etl_censo.ipynb").resolve().parent
RAW_CENSO_DIR = BASE_DIR / "data" / "raw"
RAW_ANEXOS = RAW_CENSO_DIR / "anexos"

In [3]:
df_escola_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Escola_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_matricula_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Matricula_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_turma_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Turma_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_docente_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Docente_2025.csv", sep=";", encoding="latin-1", low_memory=False)

df_escola_ac_2025 = df_escola_2025[df_escola_2025["SG_UF"] == "AC"].copy()
df_matricula_ac_2025 = df_matricula_2025[df_matricula_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()
df_turma_ac_2025 = df_turma_2025[df_turma_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()
df_docente_ac_2025 =  df_docente_2025[df_docente_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()

del df_escola_2025
del df_matricula_2025
del df_turma_2025
del df_docente_2025

gc.collect()

0

In [4]:
df_escola_ac_2025 = df_escola_ac_2025[df_escola_ac_2025["TP_SITUACAO_FUNCIONAMENTO"] == 1].copy()

In [5]:
df_censo_ac_2025 = df_escola_ac_2025.merge(
    df_matricula_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_mat")
)

df_censo_ac_2025 = df_censo_ac_2025.merge(
    df_turma_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_tur")
)

df_censo_ac_2025 = df_censo_ac_2025.merge(
    df_docente_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_doc")
)

In [6]:
del df_escola_ac_2025
del df_matricula_ac_2025
del df_turma_ac_2025
del df_docente_ac_2025

gc.collect()

0

In [7]:
df_censo_ac_2025 = df_censo_ac_2025[df_censo_ac_2025['TP_DEPENDENCIA'] == 2]

df_censo_ac_2025 = df_censo_ac_2025[SCHEMA_CURICA]

df_censo_ac_2025.shape

(592, 316)

In [8]:
df_anexos = pd.read_csv(RAW_ANEXOS / "anexos_escolas_estaduais.csv", sep=";", encoding="latin-1", low_memory=False)



In [9]:
df_anexos = df_anexos.rename(columns={'Código da Escola': 'CO_ENTIDADE'})



In [10]:
df_anexos.columns

Index([' MUNICÍPIO ', 'CO_ENTIDADE', 'ESCOLA ', 'Localização',
       'MODALIDADE DE ENSINO', 'ENDEREÇOS ', 'ATIVA(1)'],
      dtype='str')

In [11]:
df_anexos['CO_ENTIDADE'].nunique()

565

In [12]:
codigos_com_anexos = df_anexos['CO_ENTIDADE'].value_counts()
codigos_com_anexos = codigos_com_anexos[codigos_com_anexos > 1].index

df_censo_ac_2025['tem_anexo'] = df_censo_ac_2025['CO_ENTIDADE'].isin(codigos_com_anexos).astype(int)

df_censo_ac_2025['tem_anexo'].value_counts()

tem_anexo
0    519
1     73
Name: count, dtype: int64

## Teste Modelos com primeira seleção de features do SCHEMA

Foram separadas em categóricas e numéricas. 

Os modelos disponíveis são o logistic regression e random forest.

In [13]:
# ==========================================
# 1. IMPORTS
# ==========================================
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report




# ==========================================
# 2. DEFINIÇÃO DAS FEATURES
# ==========================================

cat_cols = ['TP_LOCALIZACAO', 'IN_LOCAL_FUNC_PREDIO_ESCOLAR','TP_OCUPACAO_PREDIO_ESCOLAR',
            'IN_LOCAL_FUNC_GALPAO', 'IN_LOCAL_FUNC_SALAS_OUTRA_ESC', 'IN_LOCAL_FUNC_OUTROS',
            'IN_PREDIO_COMPARTILHADO'
]

num_cols = ['QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA', 'QT_SALAS_UTILIZADAS',
            'QT_MAT_BAS', 'QT_MAT_INF', 'QT_MAT_FUND', 'QT_MAT_FUND_AI', 'QT_MAT_FUND_AF',
            'QT_MAT_MED', 'QT_MAT_MED_PROP_NS', 'QT_MAT_BAS_INT', 'QT_MAT_INF_CRE_D', 'QT_MAT_INF_CRE_DM',
            'QT_MAT_INF_CRE_DV', 'QT_MAT_INF_PRE_D', 'QT_MAT_INF_PRE_DM', 'QT_MAT_INF_PRE_DV',
            'QT_MAT_FUND_D', 'QT_MAT_FUND_DM', 'QT_MAT_FUND_DV', 'QT_MAT_MED_D', 'QT_MAT_MED_DM',
            'QT_MAT_MED_DV', 'QT_DOC_BAS', 'QT_TUR_BAS', 'QT_TUR_INF', 'QT_TUR_INF_CRE',
            'QT_TUR_INF_PRE', 'QT_TUR_FUND', 'QT_TUR_FUND_AI', 'QT_TUR_FUND_AI_1', 'QT_TUR_FUND_AI_2',
            'QT_TUR_FUND_AI_3', 'QT_TUR_FUND_AI_4', 'QT_TUR_FUND_AI_5', 'QT_TUR_FUND_AI_MULTIETAPA',
            'QT_TUR_FUND_AF', 'QT_TUR_FUND_AF_6', 'QT_TUR_FUND_AF_7', 'QT_TUR_FUND_AF_8',
            'QT_TUR_FUND_AF_9', 'QT_TUR_FUND_AF_MULTI', 'QT_TUR_FUND_AF_CORRFLUXO', 'QT_TUR_MED',
            'QT_TUR_MED_PROP', 'QT_TUR_MED_PROP_1', 'QT_TUR_MED_PROP_2', 'QT_TUR_MED_PROP_3', 'QT_TUR_MED_PROP_4',
            'QT_TUR_MED_PROP_NS', 'QT_TUR_BAS_D', 'QT_TUR_BAS_DM', 'QT_TUR_BAS_DV', 'QT_TUR_BAS_N',
            'QT_TUR_INF_CRE_D',  'QT_TUR_INF_CRE_DM', 'QT_TUR_INF_CRE_DV', 'QT_TUR_INF_CRE_N',
            'QT_TUR_INF_PRE_D', 'QT_TUR_INF_PRE_DM',  'QT_TUR_INF_PRE_DV', 'QT_TUR_INF_PRE_N', 'QT_TUR_FUND_D',
            'QT_TUR_FUND_DM', 'QT_TUR_FUND_DV', 'QT_TUR_FUND_N', 'QT_TUR_FUND_AI_D', 'QT_TUR_FUND_AI_DM',
            'QT_TUR_FUND_AI_DV', 'QT_TUR_FUND_AI_N', 'QT_TUR_FUND_AF_D', 'QT_TUR_FUND_AF_DM',
            'QT_TUR_FUND_AF_DV', 'QT_TUR_FUND_AF_N', 'QT_TUR_MED_D', 'QT_TUR_MED_DM', 'QT_TUR_MED_DV', 'QT_TUR_MED_N',
            'QT_DOC_BAS_VINCULO_CONCUR', 'QT_DOC_BAS_VINCULO_CONTRA', 'QT_DOC_BAS_VINCULO_TERCEIR', 'QT_DOC_BAS_VINCULO_CLT'
]


# ==========================================
# 3. MONTAGEM DO DATASET
# ==========================================

X = df_censo_ac_2025[cat_cols + num_cols].copy()
y = df_censo_ac_2025['tem_anexo'].copy()

# Garantia de integridade
X = X.loc[:, ~X.columns.duplicated()]


# ==========================================
# 4. SPLIT (ANTES DE QUALQUER FIT)
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# ==========================================
# 5. PIPELINES DE TRANSFORMAÇÃO
# ==========================================

# CATEGÓRICAS
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda x: x.astype(str))),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# NUMÉRICAS
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


# ==========================================
# 6. PREPROCESSADOR
# ==========================================

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_cols),
        ('num', num_pipeline, num_cols)
    ]
)


# ==========================================
# 7. MODELO
# ==========================================

#modelo = RandomForestClassifier(
#    n_estimators=200,
#    random_state=42,
#    n_jobs=-1,
#    class_weight='balanced'
#)

# modelo = LogisticRegression(
#     max_iter=1000,
#     class_weight='balanced',
#     random_state=42
# )

modelo = LogisticRegression(
    penalty='l1',
    solver='saga',   # obrigatório para L1
    C=0.1,           # controla regularização
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)

# ==========================================
# 8. PIPELINE FINAL
# ==========================================

pipeline = Pipeline(steps=[
    ('preprocessamento', preprocessador),
    ('modelo', modelo)
])


# ==========================================
# 9. TREINO
# ==========================================

pipeline.fit(X_train, y_train)


# ==========================================
# 10. AVALIAÇÃO
# ==========================================

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.92      0.95       156
           1       0.60      0.82      0.69        22

    accuracy                           0.91       178
   macro avg       0.79      0.87      0.82       178
weighted avg       0.93      0.91      0.92       178



/home/pijczs/projetos/curica_etl_censo_escolar/venv_etl_censo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/pijczs/projetos/curica_etl_censo_escolar/venv_etl_censo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


## Teste para avalição de relevância da coluna

In [25]:
# ==========================================
# 1. IMPORTS
# ==========================================
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from sklearn.inspection import permutation_importance


# ==========================================
# 2. DEFINIÇÃO DAS FEATURES
# ==========================================

cat_cols = [
    'TP_LOCALIZACAO', 'IN_LOCAL_FUNC_PREDIO_ESCOLAR','TP_OCUPACAO_PREDIO_ESCOLAR',
    'IN_LOCAL_FUNC_GALPAO', 'IN_LOCAL_FUNC_SALAS_OUTRA_ESC', 'IN_LOCAL_FUNC_OUTROS',
    'IN_PREDIO_COMPARTILHADO'
]

num_cols = [
    'QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA', 'QT_SALAS_UTILIZADAS',
    'QT_MAT_BAS', 'QT_MAT_INF', 'QT_MAT_FUND', 'QT_MAT_FUND_AI', 'QT_MAT_FUND_AF',
    'QT_MAT_MED', 'QT_MAT_MED_PROP_NS', 'QT_MAT_BAS_INT', 'QT_MAT_INF_CRE_D', 'QT_MAT_INF_CRE_DM',
    'QT_MAT_INF_CRE_DV', 'QT_MAT_INF_PRE_D', 'QT_MAT_INF_PRE_DM', 'QT_MAT_INF_PRE_DV',
    'QT_MAT_FUND_D', 'QT_MAT_FUND_DM', 'QT_MAT_FUND_DV', 'QT_MAT_MED_D', 'QT_MAT_MED_DM',
    'QT_MAT_MED_DV', 'QT_DOC_BAS', 'QT_TUR_BAS', 'QT_TUR_INF', 'QT_TUR_INF_CRE',
    'QT_TUR_INF_PRE', 'QT_TUR_FUND', 'QT_TUR_FUND_AI', 'QT_TUR_FUND_AI_1', 'QT_TUR_FUND_AI_2',
    'QT_TUR_FUND_AI_3', 'QT_TUR_FUND_AI_4', 'QT_TUR_FUND_AI_5', 'QT_TUR_FUND_AI_MULTIETAPA',
    'QT_TUR_FUND_AF', 'QT_TUR_FUND_AF_6', 'QT_TUR_FUND_AF_7', 'QT_TUR_FUND_AF_8',
    'QT_TUR_FUND_AF_9', 'QT_TUR_FUND_AF_MULTI', 'QT_TUR_FUND_AF_CORRFLUXO', 'QT_TUR_MED',
    'QT_TUR_MED_PROP', 'QT_TUR_MED_PROP_1', 'QT_TUR_MED_PROP_2', 'QT_TUR_MED_PROP_3', 'QT_TUR_MED_PROP_4',
    'QT_TUR_MED_PROP_NS', 'QT_TUR_BAS_D', 'QT_TUR_BAS_DM', 'QT_TUR_BAS_DV', 'QT_TUR_BAS_N',
    'QT_TUR_INF_CRE_D', 'QT_TUR_INF_CRE_DM', 'QT_TUR_INF_CRE_DV', 'QT_TUR_INF_CRE_N',
    'QT_TUR_INF_PRE_D', 'QT_TUR_INF_PRE_DM', 'QT_TUR_INF_PRE_DV', 'QT_TUR_INF_PRE_N',
    'QT_TUR_FUND_D', 'QT_TUR_FUND_DM', 'QT_TUR_FUND_DV', 'QT_TUR_FUND_N',
    'QT_TUR_FUND_AI_D', 'QT_TUR_FUND_AI_DM', 'QT_TUR_FUND_AI_DV', 'QT_TUR_FUND_AI_N',
    'QT_TUR_FUND_AF_D', 'QT_TUR_FUND_AF_DM', 'QT_TUR_FUND_AF_DV', 'QT_TUR_FUND_AF_N',
    'QT_TUR_MED_D', 'QT_TUR_MED_DM', 'QT_TUR_MED_DV', 'QT_TUR_MED_N',
    'QT_DOC_BAS_VINCULO_CONCUR', 'QT_DOC_BAS_VINCULO_CONTRA',
    'QT_DOC_BAS_VINCULO_TERCEIR', 'QT_DOC_BAS_VINCULO_CLT'
]


# ==========================================
# 3. DATASET
# ==========================================

X = df_censo_ac_2025[cat_cols + num_cols].copy()
y = df_censo_ac_2025['tem_anexo'].copy()

X = X.loc[:, ~X.columns.duplicated()]


# ==========================================
# 4. SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# ==========================================
# 5. PIPELINES
# ==========================================

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda x: x.astype(str))),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


# ==========================================
# 6. PREPROCESSADOR
# ==========================================

preprocessador = ColumnTransformer([
    ('cat', cat_pipeline, cat_cols),
    ('num', num_pipeline, num_cols)
])


# ==========================================
# 7. MODELO
# ==========================================

#modelo = LogisticRegression(
#    max_iter=1000,
#    class_weight='balanced',
#    random_state=42
#)

#modelo = LogisticRegression(
#    penalty='l1',
#    solver='saga',   # obrigatório para L1
#    C=0.1,           # controla regularização
#    max_iter=2000,
#    class_weight='balanced',
#    random_state=42
#)

modelo = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)


# ==========================================
# 8. PIPELINE FINAL
# ==========================================

pipeline = Pipeline([
    ('preprocessamento', preprocessador),
    ('modelo', modelo)
])


# ==========================================
# 9. TREINO
# ==========================================

pipeline.fit(X_train, y_train)


# ==========================================
# 10. AVALIAÇÃO
# ==========================================

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))


# ==========================================
# 11. PERMUTATION IMPORTANCE
# ==========================================

print("\nCalculando permutation importance...")

result = permutation_importance(
    pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# IMPORTANTE: aqui usamos X_test ORIGINAL (antes do one-hot)
df_importancia = pd.DataFrame({
    'feature': X_test.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

print("\nTop 20 variáveis mais importantes:")
print(df_importancia.head(20))

              precision    recall  f1-score   support

           0       0.94      0.99      0.96       156
           1       0.86      0.55      0.67        22

    accuracy                           0.93       178
   macro avg       0.90      0.77      0.81       178
weighted avg       0.93      0.93      0.93       178


Calculando permutation importance...

Top 20 variáveis mais importantes:
                          feature  importance
8        QT_SALAS_UTILIZADAS_FORA    0.012921
4   IN_LOCAL_FUNC_SALAS_OUTRA_ESC    0.006742
56             QT_TUR_MED_PROP_NS    0.005618
52              QT_TUR_MED_PROP_1    0.005056
81                   QT_TUR_MED_D    0.005056
27                   QT_MAT_MED_D    0.003371
15                     QT_MAT_MED    0.002809
17                 QT_MAT_BAS_INT    0.002247
51                QT_TUR_MED_PROP    0.002247
50                     QT_TUR_MED    0.002247
57                   QT_TUR_BAS_D    0.001685
47               QT_TUR_FUND_AF_9    0.001124
9

In [26]:
top_features = df_importancia.head(15)['feature'].tolist()

print(top_features)

['QT_SALAS_UTILIZADAS_FORA', 'IN_LOCAL_FUNC_SALAS_OUTRA_ESC', 'QT_TUR_MED_PROP_NS', 'QT_TUR_MED_PROP_1', 'QT_TUR_MED_D', 'QT_MAT_MED_D', 'QT_MAT_MED', 'QT_MAT_BAS_INT', 'QT_TUR_MED_PROP', 'QT_TUR_MED', 'QT_TUR_BAS_D', 'QT_TUR_FUND_AF_9', 'QT_SALAS_UTILIZADAS', 'QT_TUR_FUND_AI_D', 'QT_DOC_BAS_VINCULO_CONCUR']


## Teste com colunas relevantes

O melhor resultado foi Logistic Regression com a seleção de colunas por Random Forest.

In [30]:
# ==========================================
# 1. IMPORTS
# ==========================================
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report




# ==========================================
# 2. DEFINIÇÃO DAS FEATURES
# ==========================================

cat_cols = ['IN_LOCAL_FUNC_SALAS_OUTRA_ESC']

num_cols = ['QT_SALAS_UTILIZADAS_FORA', 'QT_TUR_MED_PROP_NS', 'QT_TUR_MED_PROP_1', 'QT_TUR_MED_D', 'QT_MAT_MED_D', 'QT_MAT_MED', 'QT_MAT_BAS_INT', 'QT_TUR_MED_PROP', 'QT_TUR_MED', 'QT_TUR_BAS_D', 'QT_TUR_FUND_AF_9', 'QT_SALAS_UTILIZADAS', 'QT_TUR_FUND_AI_D', 'QT_DOC_BAS_VINCULO_CONCUR'
]


# ==========================================
# 3. MONTAGEM DO DATASET
# ==========================================

X = df_censo_ac_2025[cat_cols + num_cols].copy()
y = df_censo_ac_2025['tem_anexo'].copy()

# Garantia de integridade
X = X.loc[:, ~X.columns.duplicated()]


# ==========================================
# 4. SPLIT (ANTES DE QUALQUER FIT)
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# ==========================================
# 5. PIPELINES DE TRANSFORMAÇÃO
# ==========================================

# CATEGÓRICAS
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda x: x.astype(str))),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# NUMÉRICAS
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


# ==========================================
# 6. PREPROCESSADOR
# ==========================================

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_cols),
        ('num', num_pipeline, num_cols)
    ]
)


# ==========================================
# 7. MODELO
# ==========================================

#modelo = LogisticRegression(
#    max_iter=1000,
#    class_weight='balanced',
#    random_state=42
#)

# Lasso - L1
modelo = LogisticRegression(
    penalty='l1',
    solver='saga',   # obrigatório para L1
    C=0.3,           # controla regularização
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)

#modelo = RandomForestClassifier(
#    n_estimators=200,
#    random_state=42,
#    n_jobs=-1,
#    class_weight='balanced'
#)


# ==========================================
# 8. PIPELINE FINAL
# ==========================================

pipeline = Pipeline(steps=[
    ('preprocessamento', preprocessador),
    ('modelo', modelo)
])


# ==========================================
# 9. TREINO
# ==========================================

pipeline.fit(X_train, y_train)


# ==========================================
# 10. AVALIAÇÃO
# ==========================================

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.96      0.97       156
           1       0.75      0.82      0.78        22

    accuracy                           0.94       178
   macro avg       0.86      0.89      0.88       178
weighted avg       0.95      0.94      0.94       178



/home/pijczs/projetos/curica_etl_censo_escolar/venv_etl_censo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/pijczs/projetos/curica_etl_censo_escolar/venv_etl_censo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


## Teste Colunas Lasso com Lasso

In [18]:
# ==========================================
# 1. IMPORTS
# ==========================================
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report




# ==========================================
# 2. DEFINIÇÃO DAS FEATURES
# ==========================================

cat_cols = ['IN_LOCAL_FUNC_SALAS_OUTRA_ESC']

num_cols = ['QT_TUR_FUND_AF_MULTI', 'QT_TUR_MED_PROP_3', 'QT_DOC_BAS_VINCULO_CONCUR', 'QT_MAT_MED_PROP_NS'
]


# ==========================================
# 3. MONTAGEM DO DATASET
# ==========================================

X = df_censo_ac_2025[cat_cols + num_cols].copy()
y = df_censo_ac_2025['tem_anexo'].copy()

# Garantia de integridade
X = X.loc[:, ~X.columns.duplicated()]


# ==========================================
# 4. SPLIT (ANTES DE QUALQUER FIT)
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# ==========================================
# 5. PIPELINES DE TRANSFORMAÇÃO
# ==========================================

# CATEGÓRICAS
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda x: x.astype(str))),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# NUMÉRICAS
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


# ==========================================
# 6. PREPROCESSADOR
# ==========================================

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_cols),
        ('num', num_pipeline, num_cols)
    ]
)


# ==========================================
# 7. MODELO
# ==========================================

modelo = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

# Lasso - L1
#modelo = LogisticRegression(
#    penalty='l1',
#    solver='saga',   # obrigatório para L1
#    C=0.1,           # controla regularização
#    max_iter=2000,
#    class_weight='balanced',
#    random_state=42
#)

#modelo = RandomForestClassifier(
#    n_estimators=200,
#    random_state=42,
#    n_jobs=-1,
#    class_weight='balanced'
#)


# ==========================================
# 8. PIPELINE FINAL
# ==========================================

pipeline = Pipeline(steps=[
    ('preprocessamento', preprocessador),
    ('modelo', modelo)
])


# ==========================================
# 9. TREINO
# ==========================================

pipeline.fit(X_train, y_train)


# ==========================================
# 10. AVALIAÇÃO
# ==========================================

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.92      0.95       156
           1       0.59      0.86      0.70        22

    accuracy                           0.91       178
   macro avg       0.79      0.89      0.83       178
weighted avg       0.93      0.91      0.92       178



## Teste de dataframe

In [31]:
# ==========================================
# 1. IMPORTS
# ==========================================
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report




# ==========================================
# 2. DEFINIÇÃO DAS FEATURES
# ==========================================

cat_cols = ['IN_LOCAL_FUNC_SALAS_OUTRA_ESC']

num_cols = ['QT_SALAS_UTILIZADAS_FORA', 'QT_TUR_MED_PROP_NS', 'QT_TUR_MED_PROP_1', 'QT_TUR_MED_D', 'QT_MAT_MED_D', 'QT_MAT_MED', 'QT_MAT_BAS_INT', 'QT_TUR_MED_PROP', 'QT_TUR_MED', 'QT_TUR_BAS_D', 'QT_TUR_FUND_AF_9', 'QT_SALAS_UTILIZADAS', 'QT_TUR_FUND_AI_D', 'QT_DOC_BAS_VINCULO_CONCUR'
]


# ==========================================
# 3. MONTAGEM DO DATASET
# ==========================================

X = df_censo_ac_2025[cat_cols + num_cols].copy()
y = df_censo_ac_2025['tem_anexo'].copy()

# Garantia de integridade
X = X.loc[:, ~X.columns.duplicated()]


# ==========================================
# 4. SPLIT (ANTES DE QUALQUER FIT)
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# ==========================================
# 5. PIPELINES DE TRANSFORMAÇÃO
# ==========================================

# CATEGÓRICAS
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda x: x.astype(str))),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# NUMÉRICAS
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


# ==========================================
# 6. PREPROCESSADOR
# ==========================================

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_cols),
        ('num', num_pipeline, num_cols)
    ]
)


# ==========================================
# 7. MODELO
# ==========================================

modelo = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

# Lasso - L1
#modelo = LogisticRegression(
#    penalty='l1',
#    solver='saga',   # obrigatório para L1
#    C=0.3,           # controla regularização
#    max_iter=2000,
#    class_weight='balanced',
#    random_state=42
#)

#modelo = RandomForestClassifier(
#    n_estimators=200,
#    random_state=42,
#    n_jobs=-1,
#    class_weight='balanced'
#)


# ==========================================
# 8. PIPELINE FINAL
# ==========================================

pipeline = Pipeline(steps=[
    ('preprocessamento', preprocessador),
    ('modelo', modelo)
])


# ==========================================
# 9. TREINO
# ==========================================

pipeline.fit(X_train, y_train)


# ==========================================
# 10. AVALIAÇÃO
# ==========================================

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.95      0.96       156
           1       0.70      0.86      0.78        22

    accuracy                           0.94       178
   macro avg       0.84      0.91      0.87       178
weighted avg       0.95      0.94      0.94       178



In [37]:
colunas_saida = [
    'NO_MUNICIPIO',
    'CO_ENTIDADE',
    'NO_ENTIDADE',
    'TP_LOCALIZACAO',
    'QT_TUR_BAS',
    'QT_SALAS_UTILIZADAS',
    'QT_SALAS_UTILIZADAS_DENTRO',
    'QT_SALAS_UTILIZADAS_FORA'
    
]

# Probabilidade da classe 1 (tem anexo)
y_prob = pipeline.predict_proba(X)[:, 1]

# Threshold (ajustável)
threshold = 0.5

y_pred = (y_prob >= threshold).astype(int)


# Resultado
df_resultado = df_censo_ac_2025[colunas_saida].copy()

df_resultado['prob_anexo'] = y_prob
df_resultado['suspeita_anexo'] = y_pred

df_resultado = df_resultado[df_resultado['NO_MUNICIPIO'] == "Cruzeiro do Sul"]

df_resultado.head(50)

# Retorna dataframe pela maior probalidade
#df_resultado = df_resultado.sort_values(
#    by='prob_anexo',
#    ascending=False
#)

,NO_MUNICIPIO,CO_ENTIDADE,NO_ENTIDADE,TP_LOCALIZACAO,QT_TUR_BAS,QT_SALAS_UTILIZADAS,QT_SALAS_UTILIZADAS_DENTRO,QT_SALAS_UTILIZADAS_FORA,prob_anexo,suspeita_anexo
151,Cruzeiro do Sul,12000094,ESC COLEGIO CRISTAO CRUZEIRO,1,21.0,11.0,11.0,0.0,0.008026,0
152,Cruzeiro do Sul,12000108,ESC PRESIDENTE TANCREDO DE ALMEIDA NEVES,1,4.0,4.0,4.0,0.0,0.033653,0
156,Cruzeiro do Sul,12000159,ESC 7 DE SETEMBRO,2,9.0,9.0,9.0,0.0,0.059088,0
157,Cruzeiro do Sul,12000167,ESC ABSOLON MOREIRA,1,13.0,10.0,10.0,0.0,0.031237,0
161,Cruzeiro do Sul,12000256,ESC ANTONIO JUVENCIO BARROSO,2,5.0,2.0,2.0,0.0,0.459153,0
163,Cruzeiro do Sul,12000280,ESC AUGUSTO SEVERO,2,7.0,6.0,6.0,0.0,0.169462,0
165,Cruzeiro do Sul,12000345,ESC CORONEL CONTREIRAS,1,9.0,5.0,5.0,0.0,0.059470,0
166,Cruzeiro do Sul,12000370,ESC CORA CORALINA,2,6.0,5.0,5.0,0.0,0.245280,0
168,Cruzeiro do Sul,12000400,ESC DOM PEDRO II,2,4.0,4.0,4.0,0.0,0.258347,0
169,Cruzeiro do Sul,12000418,ESC FLODOARDO CABRAL,1,16.0,14.0,14.0,0.0,0.000392,0
